In [ ]:
from <package> import SomeVectorStore

# === 인덱싱 ===
vectorstore = SomeVectorStore.from_documents(docs, embeddings)

# === 검색 ===
results = vectorstore.similarity_search(query, k=3)
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

# === Retriever 변환 ===
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

---

In [ ]:
# 필요한 패키지 설치 (Colab 첫 셀에서 실행)
!pip install -q langchain langchain-google-genai langchain-chroma

In [ ]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# === 임베딩 모델 (27번에서 다룬 구현체) ===
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key="YOUR_API_KEY",
)

# === 샘플 Document 리스트 ===
docs = [
    Document(
        page_content="어댑터즈는 스타트업코드에서 제공하는 개발 책 서빙 서비스입니다.",
        metadata={"source": "intro", "topic": "service"},
    ),
    Document(
        page_content="어댑터즈는 개발자 교육 콘텐츠를 제공합니다.",
        metadata={"source": "intro", "topic": "content"},
    ),
    Document(
        page_content="오늘 날씨가 정말 맑습니다.",
        metadata={"source": "weather", "topic": "weather"},
    ),
]

print(f"임베딩 객체: {type(embeddings).__name__}")
print(f"Document 수: {len(docs)}")

In [ ]:
from langchain_chroma import Chroma

# === 인메모리 모드 (persist_directory 생략) ===
vectorstore = Chroma.from_documents(docs, embeddings)

print(f"VectorStore 객체: {type(vectorstore).__name__}")
print(f"저장된 문서 수: {vectorstore._collection.count()}")

In [ ]:
results = vectorstore.similarity_search("어댑터즈는 어떤 서비스인가요?", k=2)

for i, doc in enumerate(results, 1):
    print(f"[{i}] {doc.page_content}")
    print(f"    metadata={doc.metadata}\n")

In [ ]:
results = vectorstore.similarity_search_with_score("어댑터즈는 어떤 서비스인가요?", k=3)

for i, (doc, score) in enumerate(results, 1):
    print(f"[{i}] score={score:.4f}")
    print(f"    {doc.page_content}\n")

In [ ]:
# 영구 저장 모드
import shutil, os

# === 기존 디렉토리 제거 (실습 반복용) ===
# if os.path.exists("./chroma_db"):
#     shutil.rmtree("./chroma_db")

# === 영구 저장 모드 ===
vectorstore = Chroma.from_documents(
    docs,
    embeddings,
    persist_directory="./chroma_db",
)

print("저장 완료. ./chroma_db 폴더에 SQLite 인덱스가 생성되었습니다.")
print(os.listdir("./chroma_db"))

In [ ]:
# 저장된 인덱스 다시 열기
# === 기존 인덱스 로드 (from_documents 호출 없이) ===
reopened = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,
)

results = reopened.similarity_search("어댑터즈", k=1)
print(results[0].page_content)

In [ ]:
# LCEL 체인 연결 준비
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Retriever 객체: {type(retriever).__name__}")